<a href="https://colab.research.google.com/github/jmgiorgi-10/LanguageActionTuning/blob/main/Action_Intent_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=4)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
import os
from google.colab import userdata
# Retrieve the secret and set it as an environment variable
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [56]:
from datasets import load_dataset
raw_dataset = load_dataset("json", data_files="synthetic_dataset.jsonl")
dataset = raw_dataset["train"].train_test_split(test_size=0.2, seed=42)
# Inspect the structure
print(dataset)
print("\nFirst training sample:")
print(dataset["train"][0])
print(dataset["test"][0])

DatasetDict({
    train: Dataset({
        features: ['intent', 'locale', 'user_query', 'slots'],
        num_rows: 14
    })
    test: Dataset({
        features: ['intent', 'locale', 'user_query', 'slots'],
        num_rows: 4
    })
})

First training sample:
{'intent': 'start_workout', 'locale': 'es_AR', 'user_query': 'Me gustaría empezar un entrenamiento de surf de dos horas.', 'slots': {'app_name': None, 'content': None, 'duration': '2 hours', 'recipient': None, 'spot_name': None, 'wave_metric': None, 'workout_type': 'surfing'}}
{'intent': 'open_app', 'locale': 'es_AR', 'user_query': 'Abre la app Playtomic', 'slots': {'app_name': 'Playtomic', 'content': None, 'duration': None, 'recipient': None, 'spot_name': None, 'wave_metric': None, 'workout_type': None}}


In [46]:
## compute_metrics just for action intent classification

import evaluate
import numpy as np

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    intent_logits, intent_labels = eval_pred
    intent_preds = np.argmax(intent_logits, axis=-1)

    # 1. Calculate accuracy
    acc = accuracy_score(intent_labels, intent_preds)

    # 2. Calculate Precision, Recall, and F1 all at once
    precision, recall, f1, _ = precision_recall_fscore_support(
        intent_labels,
        intent_preds,
        average="macro",
        zero_division=0
    )

    # 3. Return the clean dictionary
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [69]:
# Tokenize the dataset

from transformers import AutoTokenizer

model_checkpoint = "xlm-roberta-base" # Excellent base for multilingual (ES/EN) tasks
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Define a standard ChatML template if it's missing
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)

special_tokens_dict = {'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)

# Crucial: Resize the model embedding layer to match the new token count!
model.resize_token_embeddings(len(tokenizer))

# Extract unique intents dynamically from your dataset
unique_intents = sorted(dataset['train'].unique("intent"))

print(dataset)

# Create lookup dictionaries
intent2id = {intent: idx for idx, intent in enumerate(unique_intents)}
id2intent = {idx: intent for idx, intent in enumerate(unique_intents)}

def preprocess_function(examples):
    prompts = examples["user_query"]
    # Map the string intents to integers using our dictionary
    intent_labels = [intent2id[intent] for intent in examples["intent"]]

    formatted_texts = []
    for prompt in prompts:
        messages = [
            {"role": "user", "content": prompt}
        ]
        # Format with the model's official template
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        formatted_texts.append(text)

    # Tokenize the user prompts
    tokenized = tokenizer(
        formatted_texts,
        truncation=True,
        max_length=512,
        padding=False,
    )

    # Assign the integer IDs to labels
    tokenized["labels"] = intent_labels

    return tokenized

tokenized_dataset = dataset.map(preprocess_function, batched=True)

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['intent', 'locale', 'user_query', 'slots'],
        num_rows: 14
    })
    test: Dataset({
        features: ['intent', 'locale', 'user_query', 'slots'],
        num_rows: 4
    })
})


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['intent', 'locale', 'user_query', 'slots', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14
    })
    test: Dataset({
        features: ['intent', 'locale', 'user_query', 'slots', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
})


In [70]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"], # Standard for BERT/RoBERTa
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS # Vital for sequence classification
)

model = get_peft_model(model, peft_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [73]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

print(tokenized_dataset)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

training_arguments = TrainingArguments(
    output_dir="./results",

    # 1. Bump up the epochs
    num_train_epochs=15,              # Let it look at the data 5 times

    # 2. Fix the logging so you don't get "No log"
    logging_steps=1,                 # Log training loss every single step (since dataset is small)

    # 3. Evaluate at the end of every epoch
    eval_strategy="epoch",           # Calculate validation loss after each epoch
    save_strategy="epoch",           # Save a checkpoint after each epoch

    # Other standard settings
    learning_rate=1e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,     # Keeps the best version of the model when done
)

trainer = Trainer(
    model=model, # The adapter-wrapped sequence classification model
    args=training_arguments,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test'],
    data_collator = data_collator,
)

DatasetDict({
    train: Dataset({
        features: ['intent', 'locale', 'user_query', 'slots', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14
    })
    test: Dataset({
        features: ['intent', 'locale', 'user_query', 'slots', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
})


In [74]:
trainer.train()

## Next step --> figure out how to increase synthetic train/eval data to ~500-1000 total.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.068026,2.271191
2,1.029891,2.256889
3,1.915582,2.247906
4,1.479964,2.238570
5,1.302966,2.230810
6,1.441981,2.223709
7,1.180055,2.216705
8,1.424404,2.210620
9,1.321805,2.204996
10,1.359370,2.201125


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

TrainOutput(global_step=30, training_loss=1.3083635548750558, metrics={'train_runtime': 51.4463, 'train_samples_per_second': 4.082, 'train_steps_per_second': 0.583, 'total_flos': 2357301875520.0, 'train_loss': 1.3083635548750558, 'epoch': 15.0})